# Error Handling (Exceptions) in Python
programs meet unexpected situations : a missing file , wrong user input , a broken network , a division by zero.

    1. Syntax errors   ---> the code is not valid Python ; it does NOT run at all
    2. Exceptions      ---> the code is valid , but something goes wrong WHILE it runs

with  try / except / else / finally  we can CATCH exceptions and decide what to do ,
with  raise  we can SIGNAL our own errors.

## 1. Syntax errors vs exceptions

In [1]:
# a syntax error is found BEFORE running (when Python compiles the code)
code = "print('hello'"
try:
    compile(code, "<example>", "exec")
except SyntaxError as e:
    print("SyntaxError:", e.msg, "| line", e.lineno)

SyntaxError: '(' was never closed | line 1


In [2]:
# an exception happens WHILE running : the lines before it DO run
print("step 1")
print("step 2")
try:
    print(10 / 0)             # exception here
    print("step 3")           # never reached
except ZeroDivisionError:
    print("caught the error , program continues")
print("step 4")

step 1
step 2
caught the error , program continues
step 4


In [3]:
# without try/except an exception STOPS the program and prints a traceback
#   Traceback (most recent call last):
#     File "...", line 2, in <module>
#       result = 10 / 0
#   ZeroDivisionError: division by zero
# read a traceback from the BOTTOM : the last line tells WHAT happened ,
# the lines above tell WHERE (the chain of function calls)
def level3(x): return 10 / x
def level2(x): return level3(x) + 1
def level1(x): return level2(x) * 2

import traceback
try:
    level1(0)
except ZeroDivisionError:
    tb = traceback.format_exc().splitlines()
    print(tb[0])
    print(tb[-1])
    print("call chain:", [line.split(", in ")[-1] for line in tb if ", in " in line])

Traceback (most recent call last):
ZeroDivisionError: division by zero
call chain: ['<module>', 'level1', 'level2', 'level3']


## 2. Common built-in exceptions

In [4]:
examples = [
    ("ZeroDivisionError", lambda: 1 / 0),
    ("TypeError",         lambda: "5" + 5),
    ("ValueError",        lambda: int("abc")),
    ("IndexError",        lambda: [1, 2, 3][10]),
    ("KeyError",          lambda: {"a": 1}["b"]),
    ("AttributeError",    lambda: "text".push("x")),
    ("NameError",         lambda: undefined_variable),
    ("FileNotFoundError", lambda: open("no_such_file.csv")),
    ("ModuleNotFoundError", lambda: __import__("no_such_module")),
    ("OverflowError",     lambda: 10.0 ** 1000),
    ("RecursionError",    lambda: (f := lambda n: f(n + 1))(0)),
]
for expected, action in examples:
    try:
        action()
    except Exception as e:
        print(f"{type(e).__name__:<20} {e}")

ZeroDivisionError    division by zero
TypeError            can only concatenate str (not "int") to str
ValueError           invalid literal for int() with base 10: 'abc'
IndexError           list index out of range
KeyError             'b'
AttributeError       'str' object has no attribute 'push'
NameError            name 'undefined_variable' is not defined
FileNotFoundError    [Errno 2] No such file or directory: 'no_such_file.csv'
ModuleNotFoundError  No module named 'no_such_module'
OverflowError        (34, 'Numerical result out of range')
RecursionError       maximum recursion depth exceeded


## 3. try / except

In [5]:
# basic form
try:
    age = int("twenty")
except ValueError:
    print("please enter a number")

please enter a number


In [6]:
# as e : get the exception OBJECT (type , message , args)
try:
    age = int("twenty")
except ValueError as e:
    print("type   :", type(e).__name__)
    print("message:", e)
    print("args   :", e.args)

type   : ValueError
message: invalid literal for int() with base 10: 'twenty'
args   : ("invalid literal for int() with base 10: 'twenty'",)


In [7]:
# several except blocks : the FIRST matching one runs
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("  cannot divide by zero")
        return None
    except TypeError as e:
        print("  wrong types:", e)
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

5.0
  cannot divide by zero
None
  wrong types: unsupported operand type(s) for /: 'int' and 'str'
None


In [8]:
# one except for several exception types : use a TUPLE
def to_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return None

print([to_float(v) for v in ["3.5", "abc", None, 7, "1e-3", "۷"]])

[3.5, None, None, 7.0, 0.001, 7.0]


## 4. else and finally

In [9]:
# try    : code that may fail
# except : runs ONLY if an error happened
# else   : runs ONLY if NO error happened
# finally: runs ALWAYS (error or not) ---> cleanup
def process(text):
    print(f"process({text!r})")
    try:
        number = int(text)
    except ValueError:
        print("  except : not a number")
    else:
        print("  else   : got", number)
    finally:
        print("  finally: cleanup done")

process("42")
process("x")

process('42')
  else   : got 42
  finally: cleanup done
process('x')
  except : not a number
  finally: cleanup done


In [10]:
# why else ? keep ONLY the risky line inside try , so we don't catch errors by mistake
data = {"lr": "0.01"}
try:
    lr = float(data["lr"])
except (KeyError, ValueError) as e:
    print("bad config:", e)
else:
    print("learning rate =", lr)     # an error HERE is NOT hidden by the except above

learning rate = 0.01


In [11]:
# finally runs even when the function returns from inside try
def read_value():
    try:
        print("  opening resource")
        return 42
    finally:
        print("  finally : resource closed")

print("value =", read_value())

  opening resource
  finally : resource closed
value = 42


In [12]:
# finally runs even when the error is NOT caught
try:
    try:
        x = 1 / 0
    finally:
        print("finally runs before the error goes up")
except ZeroDivisionError:
    print("the error was caught by the OUTER try")

finally runs before the error goes up
the error was caught by the OUTER try


## 5. Exception hierarchy : catch the RIGHT thing

In [13]:
# exceptions are classes in a tree :  catching a parent also catches all its children
print(ZeroDivisionError.__mro__)
print(issubclass(ZeroDivisionError, ArithmeticError))
print(issubclass(KeyError, LookupError), issubclass(IndexError, LookupError))
print(issubclass(FileNotFoundError, OSError))
print(issubclass(KeyboardInterrupt, Exception))    # False ! it is a BaseException

(<class 'ZeroDivisionError'>, <class 'ArithmeticError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
True
True True
True
False


In [14]:
# LookupError catches BOTH KeyError and IndexError
for action in (lambda: {}["x"], lambda: [][0]):
    try:
        action()
    except LookupError as e:
        print("lookup failed:", type(e).__name__, e)

lookup failed: KeyError 'x'
lookup failed: IndexError list index out of range


In [15]:
# ORDER matters : put the SPECIFIC exception first , the general one last
def parse(value):
    try:
        return 100 / int(value)
    except ZeroDivisionError:
        return "zero is not allowed"
    except ArithmeticError:
        return "other arithmetic problem"
    except Exception as e:
        return f"unexpected: {type(e).__name__}"

print(parse("4"), "|", parse("0"), "|", parse("abc"))

25.0 | zero is not allowed | unexpected: ValueError


In [16]:
# DANGER : a too-broad except hides BUGS
def average(values):
    try:
        return sum(valeus) / len(values)      # typo : valeus  ---> NameError
    except Exception:
        return 0                              # the bug is hidden ! we always get 0

print("average:", average([1, 2, 3]), "  <--- wrong and silent")

def average_ok(values):
    try:
        return sum(values) / len(values)
    except ZeroDivisionError:                 # catch ONLY what you expect
        return 0

print("average:", average_ok([1, 2, 3]), average_ok([]))

average: 0   <--- wrong and silent
average: 2.0 0


In [17]:
# never use a bare  except:  it even catches KeyboardInterrupt (Ctrl+C) and SystemExit
try:
    raise KeyboardInterrupt            # simulate the user pressing Ctrl+C / Interrupt
except Exception:
    print("Exception does NOT catch it")
except BaseException as e:
    print("only BaseException catches it:", type(e).__name__)

only BaseException catches it: KeyboardInterrupt


## 6. raise : signal your own errors

In [18]:
def set_learning_rate(lr):
    if not isinstance(lr, (int, float)):
        raise TypeError(f"lr must be a number, got {type(lr).__name__}")
    if not 0 < lr < 1:
        raise ValueError(f"lr must be in (0, 1), got {lr}")
    return lr

for value in (0.001, 5, "fast"):
    try:
        print("ok:", set_learning_rate(value))
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

ok: 0.001
ValueError: lr must be in (0, 1), got 5
TypeError: lr must be a number, got str


In [19]:
# re-raise : do something (log) and let the error continue upwards
def load(path):
    try:
        return open(path).read()
    except FileNotFoundError:
        print(f"  [log] file missing: {path}")
        raise                        # same exception , same traceback

try:
    load("missing.csv")
except FileNotFoundError as e:
    print("caller handles it:", e.filename)

  [log] file missing: missing.csv
caller handles it: missing.csv


In [20]:
# raise ... from ... : exception CHAINING (keep the original cause)
def read_config(text):
    import json
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError("config file is not valid JSON") from e

try:
    read_config("{lr: 0.01}")          # keys must be in double quotes
except ValueError as e:
    print("error :", e)
    print("cause :", type(e.__cause__).__name__, "-", e.__cause__)

error : config file is not valid JSON
cause : JSONDecodeError - Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


In [21]:
# add_note (Python 3.11+) : attach extra context to an exception
try:
    try:
        int("12a")
    except ValueError as e:
        e.add_note("while reading column 'price' , row 17")
        raise
except ValueError as e:
    print(e)
    print(e.__notes__)

invalid literal for int() with base 10: '12a'
["while reading column 'price' , row 17"]


## 7. Custom exceptions

In [22]:
# create a small hierarchy for YOUR domain (inherit from Exception)
class DataError(Exception):
    """Base class for all data problems in our pipeline."""

class MissingColumnError(DataError):
    def __init__(self, missing):
        self.missing = missing
        super().__init__(f"missing columns: {sorted(missing)}")

class InvalidValueError(DataError):
    pass

def validate(row, required=("age", "income")):
    missing = set(required) - row.keys()
    if missing:
        raise MissingColumnError(missing)
    if row["age"] < 0:
        raise InvalidValueError(f"age cannot be negative: {row['age']}")
    return True

for row in [{"age": 30, "income": 5000}, {"age": 30}, {"age": -4, "income": 100}]:
    try:
        validate(row)
        print("valid  :", row)
    except MissingColumnError as e:
        print("missing:", e, "| attribute:", e.missing)
    except DataError as e:                       # catches every other data problem
        print("invalid:", type(e).__name__, "-", e)

valid  : {'age': 30, 'income': 5000}
missing: missing columns: ['income'] | attribute: {'income'}
invalid: InvalidValueError - age cannot be negative: -4


## 8. EAFP vs LBYL

In [23]:
# LBYL : Look Before You Leap  ---> check first
config = {"batch_size": 32}
if "lr" in config:
    lr = config["lr"]
else:
    lr = 0.001

# EAFP : Easier to Ask Forgiveness than Permission ---> try , then handle (pythonic)
try:
    lr = config["lr"]
except KeyError:
    lr = 0.001
print(lr)

# for dicts the cleanest is often .get()
print(config.get("lr", 0.001))

0.001
0.001


In [24]:
# EAFP is better when the check is hard or incomplete
def is_number_lbyl(s):
    return s.replace(".", "", 1).isdigit()      # misses "-3", "1e-3", " 4 " ...

def is_number_eafp(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

for s in ["3.5", "-3", "1e-3", " 4 ", "abc", "1.2.3"]:
    print(f"{s!r:8} LBYL={is_number_lbyl(s)!s:5}  EAFP={is_number_eafp(s)}")

'3.5'    LBYL=True   EAFP=True
'-3'     LBYL=False  EAFP=True
'1e-3'   LBYL=False  EAFP=True
' 4 '    LBYL=False  EAFP=True
'abc'    LBYL=False  EAFP=False
'1.2.3'  LBYL=False  EAFP=False


## 9. assert : check things that must ALWAYS be true (for developers)

In [25]:
def accuracy(y_true, y_pred):
    assert len(y_true) == len(y_pred), "y_true and y_pred must have the same length"
    assert len(y_true) > 0, "empty input"
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

print(accuracy([1, 0, 1], [1, 1, 1]))
try:
    accuracy([1, 0, 1], [1, 1])
except AssertionError as e:
    print("AssertionError:", e)

0.6666666666666666
AssertionError: y_true and y_pred must have the same length


In [26]:
# WARNING : asserts are REMOVED when Python runs with  -O  (optimize)
#   python -O train.py   ---> all assert lines are skipped
# so NEVER use assert to validate user input or data from outside ---> use raise
import sys
print("asserts active:", __debug__)

asserts active: True


## 10. Context managers : with  (automatic cleanup)

In [27]:
# with open(...) closes the file EVEN IF an error happens inside the block
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("line 1\nline 2\n")
print("closed after with:", f.closed)

try:
    with open("demo.txt", encoding="utf-8") as f:
        first = f.readline()
        raise ValueError("something failed while reading")
except ValueError as e:
    print("error:", e, "| file closed:", f.closed)

closed after with: True
error: something failed while reading | file closed: True


In [28]:
# write your own context manager with contextlib
from contextlib import contextmanager
import time

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield                                 # the body of the with-block runs here
    finally:
        print(f"[{label}] {(time.perf_counter() - start) * 1000:.1f} ms")

with timer("sum of squares"):
    total = sum(i * i for i in range(500_000))
print(total)

[sum of squares] 22.6 ms
41666541666750000


In [29]:
# contextlib.suppress : ignore one SPECIFIC expected exception
from contextlib import suppress
import os

with suppress(FileNotFoundError):
    os.remove("demo.txt")                     # delete the demo file
with suppress(FileNotFoundError):
    os.remove("demo.txt")                     # already deleted ---> ignored silently
print("demo file exists:", os.path.exists("demo.txt"))

demo file exists: False


## 11. Warnings : problems that should NOT stop the program

In [30]:
import warnings

def fit(X, max_iter=3):
    for i in range(max_iter):
        pass
    converged = False
    if not converged:
        warnings.warn(f"did not converge after {max_iter} iterations", RuntimeWarning)
    return "model"

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = fit([1, 2, 3])
print("result:", model)
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

result: model


In [31]:
# turn warnings into errors (useful in tests / strict pipelines)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    try:
        fit([1, 2, 3])
    except RuntimeWarning as e:
        print("now it is an error:", e)

now it is an error: did not converge after 3 iterations


## 12. Exception groups (Python 3.11+) : many errors at once

In [32]:
def validate_all(rows):
    errors = []
    for i, row in enumerate(rows):
        if row.get("age", 0) < 0:
            errors.append(ValueError(f"row {i}: negative age"))
        if "income" not in row:
            errors.append(KeyError(f"row {i}: missing income"))
    if errors:
        raise ExceptionGroup("validation failed", errors)

try:
    validate_all([{"age": -1, "income": 10}, {"age": 5}, {"age": -3}])
except* ValueError as group:
    print("value errors:", [str(e) for e in group.exceptions])
except* KeyError as group:
    print("key errors  :", [str(e) for e in group.exceptions])

value errors: ['row 0: negative age', 'row 2: negative age']
key errors  : ["'row 1: missing income'", "'row 2: missing income'"]


## 13. Applications in AI , Machine Learning and Expert Systems

In [33]:
# Application 1 : robust data loading ---> skip bad rows , but COUNT and REPORT them
raw_rows = ["25,50000", "31,62000", "abc,40000", "45,", "29,48000", "52,91000,extra"]
clean, errors = [], []
for line_no, line in enumerate(raw_rows, start=1):
    try:
        age_s, income_s = line.split(",")          # ValueError if not exactly 2 fields
        clean.append((int(age_s), float(income_s)))
    except ValueError as e:
        errors.append((line_no, line, str(e)))

print("clean rows:", clean)
print(f"rejected {len(errors)}/{len(raw_rows)} rows:")
for line_no, line, msg in errors:
    print(f"  line {line_no}: {line!r:18} -> {msg}")

clean rows: [(25, 50000.0), (31, 62000.0), (29, 48000.0)]
rejected 3/6 rows:
  line 3: 'abc,40000'        -> invalid literal for int() with base 10: 'abc'
  line 4: '45,'              -> could not convert string to float: ''
  line 6: '52,91000,extra'   -> too many values to unpack (expected 2)


In [34]:
# Application 2 : parse JSON returned by an LLM (often wrapped in text or broken)
import json, re

def parse_llm_json(text):
    try:
        return json.loads(text)                       # 1) clean JSON
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, re.DOTALL)     # 2) JSON inside extra text
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError as e:
            raise ValueError(f"found braces but invalid JSON: {e.msg}") from e
    raise ValueError("no JSON object in model output")

outputs = [
    '{"label": "positive", "score": 0.93}',
    'Sure! Here is the result:\n{"label": "negative", "score": 0.81}\nHope it helps.',
    '{"label": "neutral", score: 0.5}',
    "I cannot classify this text.",
]
for out in outputs:
    try:
        print("parsed:", parse_llm_json(out))
    except ValueError as e:
        print("failed:", e)

parsed: {'label': 'positive', 'score': 0.93}
parsed: {'label': 'negative', 'score': 0.81}
failed: found braces but invalid JSON: Expecting property name enclosed in double quotes
failed: no JSON object in model output


In [35]:
# Application 3 : retry with exponential backoff for a flaky API (simulated)
import random, time
random.seed(7)

class RateLimitError(Exception):
    pass

def call_model_api(prompt):
    if random.random() < 0.6:                         # 60% of calls fail
        raise RateLimitError("429 Too Many Requests")
    return f"answer to {prompt!r}"

def call_with_retry(prompt, max_attempts=5, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return call_model_api(prompt)
        except RateLimitError as e:
            if attempt == max_attempts:
                raise RuntimeError(f"gave up after {max_attempts} attempts") from e
            delay = base_delay * 2 ** (attempt - 1)
            print(f"  attempt {attempt}: {e} ---> wait {delay * 1000:.0f} ms")
            time.sleep(delay)

print(call_with_retry("summarize this report"))

  attempt 1: 429 Too Many Requests ---> wait 10 ms
  attempt 2: 429 Too Many Requests ---> wait 20 ms
answer to 'summarize this report'


In [36]:
# Application 4 : validate the input of a model-serving API BEFORE predicting
FEATURES = {"age": (0, 120), "income": (0, 10_000_000), "tenure": (0, 600)}

class InvalidRequest(Exception):
    pass

def predict_request(payload: dict) -> dict:
    missing = FEATURES.keys() - payload.keys()
    if missing:
        raise InvalidRequest(f"missing features: {sorted(missing)}")
    for name, (lo, hi) in FEATURES.items():
        value = payload[name]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise InvalidRequest(f"{name} must be a number, got {value!r}")
        if not lo <= value <= hi:
            raise InvalidRequest(f"{name}={value} outside [{lo}, {hi}]")
    score = min(1.0, payload["income"] / 1_000_000)      # dummy model
    return {"status": 200, "score": round(score, 3)}

requests = [
    {"age": 35, "income": 450_000, "tenure": 24},
    {"age": 35, "income": 450_000},
    {"age": "35", "income": 450_000, "tenure": 24},
    {"age": 250, "income": 450_000, "tenure": 24},
]
for req in requests:
    try:
        print(predict_request(req))
    except InvalidRequest as e:
        print({"status": 422, "error": str(e)})

{'status': 200, 'score': 0.45}
{'status': 422, 'error': "missing features: ['tenure']"}
{'status': 422, 'error': "age must be a number, got '35'"}
{'status': 422, 'error': 'age=250 outside [0, 120]'}


In [37]:
# Application 5 : stop training when the loss becomes NaN / inf , and save a checkpoint on Ctrl+C
import math

class TrainingDiverged(Exception):
    pass

def train(losses, interrupt_at=None):
    best = None
    try:
        for epoch, loss in enumerate(losses, start=1):
            if epoch == interrupt_at:
                raise KeyboardInterrupt               # simulate the user pressing Stop
            if not math.isfinite(loss):
                raise TrainingDiverged(f"loss={loss} at epoch {epoch} (try a smaller learning rate)")
            best = loss if best is None else min(best, loss)
            print(f"  epoch {epoch}: loss={loss}")
    except KeyboardInterrupt:
        print("  interrupted by user ---> saving checkpoint")
    finally:
        print(f"  [finally] best loss so far = {best}")
    return best

print("run 1:"); train([0.9, 0.6, 0.5])
print("run 2:")
try:
    train([0.9, 3.7, float("nan")])
except TrainingDiverged as e:
    print("  TrainingDiverged:", e)
print("run 3:"); train([0.9, 0.6, 0.5, 0.4], interrupt_at=3)

run 1:
  epoch 1: loss=0.9
  epoch 2: loss=0.6
  epoch 3: loss=0.5
  [finally] best loss so far = 0.5
run 2:
  epoch 1: loss=0.9
  epoch 2: loss=3.7
  [finally] best loss so far = 0.9
  TrainingDiverged: loss=nan at epoch 3 (try a smaller learning rate)
run 3:
  epoch 1: loss=0.9
  epoch 2: loss=0.6
  interrupted by user ---> saving checkpoint
  [finally] best loss so far = 0.6


In [38]:
# Application 6 : expert system that ASKS for missing facts instead of crashing
class UnknownFact(Exception):
    def __init__(self, fact):
        self.fact = fact
        super().__init__(f"unknown fact: {fact}")

def get(facts, name):
    try:
        return facts[name]
    except KeyError:
        raise UnknownFact(name) from None             # hide the internal KeyError

RULES = [
    ("bearing failure", lambda f: get(f, "temp") > 85 and get(f, "vibration") > 7.1),
    ("cooling problem", lambda f: get(f, "temp") > 85),
]

def diagnose(facts, answers):
    while True:
        try:
            return [name for name, cond in RULES if cond(facts)] or ["normal"]
        except UnknownFact as e:
            print(f"  system asks: what is {e.fact!r}?  ---> {answers[e.fact]}")
            facts[e.fact] = answers[e.fact]           # in a real system: input(...)

print(diagnose({"temp": 92}, answers={"vibration": 8.0}))

  system asks: what is 'vibration'?  ---> 8.0
['bearing failure', 'cooling problem']


## Exercises
1. write `safe_int(text, default=None)` that also accepts Persian digits and spaces , and returns `default` for invalid input.
2. predict what is printed , and in which ORDER , for `x = 2 / 1` and then for `x = 2 / 0` :
   `try:` print(1) and `x = ...` , `except ZeroDivisionError:` print(2) , `else:` print(3) , `finally:` print(4).
3. why is `except Exception: return 0` in `average` dangerous ? fix it in two different ways.
4. create `class ModelNotTrainedError(Exception)` and raise it from a `predict()` function when `model is None`.
5. change Application 1 so that it ALSO rejects rows where age > 120 , with a clear error message.
6. in Application 2 , add a third strategy : remove ```json fences before parsing.
7. write a context manager `@contextmanager def working_directory(path)` that changes the folder and ALWAYS goes back.
8. explain the difference between  `raise`  ,  `raise e`  ,  `raise NewError() from e`  and  `raise NewError() from None`.